# Community Notes 陣営反応分析 — Simple H1 版 (方向別 4 ダミー)

このノートは `colab_simple.ipynb` と並列に動く H1 検証版です。
**既存の simple_*.csv 系ファイルは上書きしません** (別名 `simple_h1_*` で書き出し)。

## H1 の狙い
元 simple の回帰で `β_typeA = -1.13 (p<0.001)` となり、「TypeA バーストがあるとむしろ採用されやすい」という
元仮説とは逆の結果が出た。これはバーストが NOT_HELPFUL 一斉ではなく **HELPFUL 一斉** が多いことに
由来する可能性がある。H1 ではバースト方向 (`helpful` / `nothelp`) で 4 ダミーに切り直して再推定する。

## 回帰式 (H1)
```
deleted ~ type_a_helpful + type_a_nothelp + type_b_helpful + type_b_nothelp
        + quality + log_ratings_count
```

## H1 の判定
- **`β(type_a_nothelp) > 0` で有意** → 元仮説「陣営反応で潰されている」を支持
- **`β(type_a_helpful) < 0` で有意** → 元 TypeA 負係数の正体は『擁護一斉』
- 両者非有意 → H1 不支持 / サンプル不足

## 実行時間の目安
| SAMPLE_FRAC | 所要時間 | 用途 |
|---|---|---|
| 0.05 | 3〜5 分 | 動作確認 |
| 0.30 | 10〜15 分 | 本番 |
| 1.00 | 30〜60 分 | フル（メモリ注意）|

---
## ★ 設定（ここを編集して実行を調整）

In [ ]:
# ====================================================
# ★ サンプリング
# ====================================================
# 頑健性チェック: SEED を 1, 2, 3 と変えて同じ frac で再実行し、
# β(type_a_helpful) / β(type_a_nothelp) の符号・有意性がブレないかを見る。

SAMPLE_FRAC = 0.30    # 0.05 で動作確認、0.30 で本番
SEED        = 42

# ====================================================
# ★ Polarity / Burst (simple と同じ既定)
# ====================================================
POLARITY_FIRST_N = 50
BURST_MULTIPLIER = 3.0
BURST_MIN_COUNT  = 5

# ====================================================
# ★ データ場所 (Drive 上の zip フォルダ)
# ====================================================
DRIVE_DATA = '/content/drive/Shareddrives/基礎プロジェクト/data'

# ====================================================
# ★ 結果の保存先 — simple とは別ディレクトリにして上書きを防ぐ
# ====================================================
SAVE_DIR = '/content/drive/MyDrive/toriumi_x3_results_simple_h1'

# ====================================================
# ★ GitHub
# ====================================================
REPO_URL = 'https://github.com/hirototoda/toriumi_x3.git'
REPO_DIR = '/content/toriumi_x3'

---
## 0. セットアップ（以下は変更不要）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

if os.path.exists(DRIVE_DATA):
    print(f'OK: フォルダが見つかりました → {DRIVE_DATA}')
    print()
    print('中身:')
    for d in sorted(os.listdir(DRIVE_DATA)):
        print(f'  {d}')
else:
    print(f'ERROR: {DRIVE_DATA} が見つかりません')
    print()
    sd = '/content/drive/Shareddrives'
    if os.path.exists(sd):
        print('利用できる共有ドライブ:')
        for d in os.listdir(sd):
            print(f'  {d}')
    print('MyDrive (上位 20 件):')
    for d in sorted(os.listdir('/content/drive/MyDrive'))[:20]:
        print(f'  {d}')

In [ ]:
# GitHub から clone + pull + 依存インストール
!git clone {REPO_URL} {REPO_DIR} 2>/dev/null || echo 'already cloned'
os.chdir(REPO_DIR)
!git pull
!pip install -q statsmodels

In [ ]:
# Drive 上の zip を Colab ローカル (data/raw/) で解凍。
# 既に TSV が存在すればスキップ (再実行安全)。
#
# Drive 側のサブフォルダ名はスペース有無や大小が表記ゆれしているため、
# 候補リストから先に存在するものを採用する。

import zipfile

raw_dir = f'{REPO_DIR}/data/raw'
os.makedirs(raw_dir, exist_ok=True)

def _unzip_all(src_folder, prefix):
    if not os.path.exists(src_folder):
        print(f'  [skip] {src_folder} が無い')
        return
    zips = sorted(f for f in os.listdir(src_folder) if f.startswith(prefix) and f.endswith('.zip'))
    if not zips:
        print(f'  [skip] {prefix}-*.zip が {src_folder} に無い')
        return
    for f in zips:
        tsv_name = f.replace('.zip', '.tsv')
        if os.path.exists(os.path.join(raw_dir, tsv_name)):
            print(f'  [skip exists] {tsv_name}')
            continue
        print(f'  [unzip] {f}')
        with zipfile.ZipFile(os.path.join(src_folder, f)) as zf:
            zf.extractall(raw_dir)

def _pick_subfolder(candidates):
    for name in candidates:
        full = os.path.join(DRIVE_DATA, name)
        if os.path.exists(full):
            return full, name
    return None, candidates[0]  # 見つからなくても最後にメッセージ用に先頭を返す

# (zip prefix, Drive 上のサブフォルダ候補) — スペース有無の表記ゆれを吸収
TARGETS = [
    ('ratings',           ['RatingData', 'Rating data', 'ratings']),
    ('notes',             ['Notes data', 'NotesData', 'notes']),
    ('noteStatusHistory', ['Note status history data', 'NoteStatusHistoryData', 'noteStatusHistory']),
]

for prefix, candidates in TARGETS:
    folder, used = _pick_subfolder(candidates)
    label = used if folder else f'{candidates[0]} (見つからず)'
    print(f'\n● {label} ({prefix}-*)')
    if folder:
        _unzip_all(folder, prefix)
    else:
        tried = ' / '.join(candidates)
        print(f'  [error] {DRIVE_DATA} 直下に {tried} のいずれも存在しない')

print('\n--- data/raw の中身 ---')
for f in sorted(os.listdir(raw_dir)):
    size_mb = os.path.getsize(os.path.join(raw_dir, f)) / (1024 * 1024)
    print(f'  {f:50s} {size_mb:8.1f} MB')


---
## 1. パイプライン実行 (H1 版)

simple 版との差分:
- **バースト**: 方向 (`helpful` / `nothelp` / `mixed`) を計算し、`mixed` は除外
- **回帰**: `type_a / type_b` を `{helpful, nothelp}` で 2 分割 → 4 ダミー
- **出力**: `data/processed/simple_h1_*.csv` `simple_h1_regression.txt` (simple の出力とは別ファイル)

実行時間: SAMPLE_FRAC=0.30 で 10〜15 分目安。

In [ ]:
import subprocess, sys, os

os.chdir(REPO_DIR)

proc = subprocess.run(
    [sys.executable, 'scripts/experiments/run_simple_h1.py',
     '--sample-frac',      str(SAMPLE_FRAC),
     '--seed',             str(SEED),
     '--polarity-first-n', str(POLARITY_FIRST_N),
     '--burst-multiplier', str(BURST_MULTIPLIER),
     '--burst-min-count',  str(BURST_MIN_COUNT)],
    check=False,
)

if proc.returncode != 0:
    raise RuntimeError(
        f'run_simple_h1.py が exit code {proc.returncode} で失敗しました。'
        ' 上のスクリプト出力 (Traceback など) を確認してください。'
    )

expected = 'data/processed/simple_h1_features.csv'
if not os.path.exists(expected):
    print('[debug] data/processed の中身:')
    if os.path.isdir('data/processed'):
        for f in sorted(os.listdir('data/processed')):
            sz = os.path.getsize(f'data/processed/{f}')
            print(f'  {f}  ({sz:,} bytes)')
    else:
        print('  (data/processed ディレクトリ自体が無い)')
    raise RuntimeError(
        f'{expected} が生成されていません。'
        ' 上のスクリプト出力を確認してください。'
    )

print(f'\nOK: {expected} ({os.path.getsize(expected):,} bytes) 生成完了')

---
## 2. 結果の確認

In [ ]:
# 2-1. 特徴量テーブル (H1: 4 ダミー版)
import os
import pandas as pd

os.chdir(REPO_DIR)

feat = pd.read_csv('data/processed/simple_h1_features.csv')
print(f'features: {len(feat):,} notes')
print(f"  deleted (Not Helpful) = {int(feat['deleted'].sum())}")
print(f"  helpful               = {int((feat['deleted']==0).sum())}")
print()
print('  方向別バースト件数:')
print(f"    TypeA + helpful    = {int(feat['type_a_helpful'].sum())}")
print(f"    TypeA + nothelp    = {int(feat['type_a_nothelp'].sum())}")
print(f"    TypeB + helpful    = {int(feat['type_b_helpful'].sum())}")
print(f"    TypeB + nothelp    = {int(feat['type_b_nothelp'].sum())}")
print(f"    (burst なし)       = {int(((feat[['type_a_helpful','type_a_nothelp','type_b_helpful','type_b_nothelp']].sum(axis=1))==0).sum())}")
display(feat.head(10))

In [ ]:
# 2-2. バースト一覧 (方向付き)
burst_path = 'data/processed/simple_h1_bursts.csv'
if os.path.exists(burst_path):
    bursts = pd.read_csv(burst_path)
    print(f'bursts: {len(bursts):,}')
    print('\nTypeA/B 分布:')
    print(bursts['burst_type'].value_counts(dropna=False))
    print('\n方向分布:')
    print(bursts['burst_direction'].value_counts(dropna=False))
    print('\nTypeA/B × 方向 クロス:')
    print(pd.crosstab(bursts['burst_type'], bursts['burst_direction']))
    display(bursts.head())
else:
    print(f'{burst_path} が無い (バースト検出 0 件)')

In [ ]:
# 2-3. 回帰 summary (テキスト全文) — H1 判定も自動 print 済み
# β(type_a_nothelp), β(type_a_helpful) の符号・有意性を確認する。
print(open('data/processed/simple_h1_regression.txt').read())

---
## 3. 結果を Drive に保存

In [ ]:
# simple_h1_*.csv と simple_h1_*.txt を SAVE_DIR にコピー
# (simple_*.csv は colab_simple.ipynb 側で既に別ディレクトリに保存されている想定)
import shutil

os.makedirs(SAVE_DIR, exist_ok=True)
copied = 0
for f in os.listdir('data/processed'):
    if f.startswith('simple_h1_'):
        shutil.copy(os.path.join('data/processed', f), SAVE_DIR)
        print(f'  copied: {f}')
        copied += 1
print(f'\nDone! {copied} ファイルを保存: {SAVE_DIR}')

---
## おまけ: 頑健性チェック

同じ `SAMPLE_FRAC` で `SEED` を 1, 2, 3 と変えて 3 回実行し、
`β(type_a_nothelp)` と `β(type_a_helpful)` の符号・有意性がブレないか確認する。

1. 上の「★ 設定」セルで `SEED = 1` に変更
2. セル 1 (パイプライン実行) だけ再実行
3. 同様に `SEED = 2`, `SEED = 3` で実行
4. 3 回の結果で判定が一貫していれば頑健

※ 出力ファイルは毎回上書きされるので、比較したいときは手動で save dir を分けてください。